In [53]:
from dataclasses import dataclass
import numpy as np 
import torch
import matplotlib.pyplot as plt 
import gymnasium as gym
import torch.nn.functional as F
import copy
from stable_baselines3 import PPO,SAC
from src.pcgym.pcgym import make_env
import jax.numpy as jnp
#Global params
T = 26
nsteps = 100

In [51]:
T = 26
nsteps = 60

SP = {
    'CA6': [0.5 for i in range(int(nsteps/3))] + 
          [0.0 for i in range(int(nsteps/3))] + 
          [0.0 for i in range(int(nsteps/3))],
}

action_space = {
    'low': np.array([0.3, 0.3]),
    'high': np.array([2.0, 2.0])
}

observation_space = {
    'low': np.array( [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),
    'high': np.array([2.0,2.0, 2.0, 2.0, 2.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]),
}

r_scale = {'CA6': 1e3}

env_params = {
    'N': nsteps,
    'tsim': T,
    'SP': SP,
    'o_space': observation_space,
    'a_space': action_space,
    'x0': np.array([1.0]*6 + [0.4]*6),
    'r_scale': r_scale,
    'model': 'pfr',
    'normalise_a': True,
    'normalise_o': True,
    'noise': True,
    'integration_method': 'casadi',
    'noise_percentage': 0.001,
}
env = make_env(env_params)

In [44]:
env.model.info()

{'parameters': {'Peh': 5.0,
  'Pem': 5.0,
  'Le': 1.0,
  'Da': 0.875,
  'gamma': 15.0,
  'eta': 0.8375,
  'mu': 13.0,
  'Tw0': 1.0,
  'T0': 1.0,
  'CA0': 1.0,
  'deltaz': 0.2,
  'states': ['T1',
   'T2',
   'T3',
   'T4',
   'T5',
   'T6',
   'CA1',
   'CA2',
   'CA3',
   'CA4',
   'CA5',
   'CA6'],
  'inputs': ['Tw1', 'Tw2'],
  'disturbances': ['T0', 'CA0'],
  'uncertainties': None},
 'states': ['T1',
  'T2',
  'T3',
  'T4',
  'T5',
  'T6',
  'CA1',
  'CA2',
  'CA3',
  'CA4',
  'CA5',
  'CA6'],
 'inputs': ['Tw1', 'Tw2'],
 'disturbances': ['T0', 'CA0'],
 'uncertainties': []}

In [45]:
class p_controller:
    def predict(o, deterministic = False):
        sp = o[2]
        x = o[0]
        e = sp - x
        kp = -0.1
        u = kp*e
        return u, 0

In [46]:
class pi_controller:
  integral = 0

  @staticmethod
  def predict(o, deterministic=False):
    sp = o[2]
    x = o[0]
    e = sp - x
    kp = -0.1
    ki = -0.01
    pi_controller.integral += e
    u = kp * e + ki * pi_controller.integral
    return u, 0

In [47]:
class pid_controller:
  integral = 0
  previous_error = 0

  @staticmethod
  def predict(o, deterministic=False):
    sp = o[2]
    x = o[0]
    e = sp - x
    kp = -0.1
    ki = -0.01
    kd = -0
    pid_controller.integral += e
    derivative = e - pid_controller.previous_error
    u = kp * e + ki * pid_controller.integral + kd * derivative
    pid_controller.previous_error = e
    return u, 0

In [49]:
evaluator, data = env.plot_rollout({'P': p_controller, 'PI':pi_controller, 'PID':pid_controller}, reps=10)

ValueError: could not broadcast input array from shape (12,) into shape (13,)